# Head Circuit Decomposition

Decomposes attention head circuits into interpretable subcircuits:
QK circuit analysis, OV circuit decomposition, virtual attention heads,
and composition patterns.

**References:**
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"
- Olsson et al. (2022) "In-context Learning and Induction Heads"

## Why This Matters

Composition head circuit decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_circuit_decomposition import (
    qk_circuit_analysis,
    ov_circuit_analysis,
    virtual_attention_head,
    head_composition_pattern,
    head_logit_contribution,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads, d_model={cfg.d_model}")

In [ ]:
# 1. QK circuit analysis - what tokens attend to what
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        qk = qk_circuit_analysis(model, layer=l, head=h)
        print(f"L{l}H{h}: rank={qk['effective_rank']:.0f}, top SVs={qk['eigenvalues'][:3].tolist()}")

In [ ]:
# 2. OV circuit analysis - what information is moved
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        ov = ov_circuit_analysis(model, layer=l, head=h)
        print(f"L{l}H{h}: rank={ov['effective_rank']:.0f}, trace={ov['trace']:+.4f}, "
              f"top SVs={ov['singular_values'][:3].tolist()}")

In [ ]:
# 3. Virtual attention heads - composition of head pairs
print("Virtual heads (L0->L1):")
for ha in range(cfg.n_heads):
    for hb in range(cfg.n_heads):
        vh = virtual_attention_head(model, 0, ha, 1, hb)
        print(f"  L0H{ha}->L1H{hb}: QK={vh['qk_composition']:.4f}, OV={vh['ov_composition']:.4f}, "
              f"total={vh['composition_strength']:.4f}")

In [ ]:
# 4. Composition pattern across all head pairs
comp = head_composition_pattern(model)
print(f"Strongest QK composition: L{comp['strongest_qk_pair'][0]}H{comp['strongest_qk_pair'][1]} -> "
      f"L{comp['strongest_qk_pair'][2]}H{comp['strongest_qk_pair'][3]}")
print(f"Strongest OV composition: L{comp['strongest_ov_pair'][0]}H{comp['strongest_ov_pair'][1]} -> "
      f"L{comp['strongest_ov_pair'][2]}H{comp['strongest_ov_pair'][3]}")
print(f"Mean composition: {comp['mean_composition']:.4f}")

In [ ]:
# 5. Head logit contribution - what tokens does each head promote?
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        hlc = head_logit_contribution(model, tokens, layer=l, head=h)
        promoted = [f"t{t}({v:+.3f})" for t, v in hlc['top_promoted'][:3]]
        demoted = [f"t{t}({v:+.3f})" for t, v in hlc['top_demoted'][:3]]
        print(f"L{l}H{h}: norm={hlc['output_norm']:.4f}, "
              f"promotes=[{', '.join(promoted)}], demotes=[{', '.join(demoted)}]")